In [ ]:
SHEET_ID = "12YxYgetu4pyqHdbT0NL1NBFiCdGJEo7N"

# paste gids from your links
GIDS = {
    "atm_registry": "2053419249",
    "atm_history": "913700150",
    "kassa_registry": "1755061160",
    "kassa_history": "1963259370"
}

# ================================
# 1. LOAD DATA
# ================================
import pandas as pd

def load_sheet(sheet_id, gid):
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    return pd.read_csv(url)

atm_registry = load_sheet(SHEET_ID, GIDS["atm_registry"])
atm_history = load_sheet(SHEET_ID, GIDS["atm_history"])
kassa_registry = load_sheet(SHEET_ID, GIDS["kassa_registry"])
kassa_history = load_sheet(SHEET_ID, GIDS["kassa_history"])

print("Data loaded")

Data loaded


In [ ]:
!pip -q install scikit-learn

import numpy as np
from sklearn.cluster import DBSCAN

In [ ]:
import pandas as pd

# Define 'status' DataFrame by copying 'atm_registry' as it contains 'Lat' and 'Lon'
# and is a suitable base for service needs assessment.
status = atm_registry.copy()

# Add a placeholder 'Needs_Service' column.
# In a real application, this column would be populated based on specific business rules
# (e.g., ATM cash levels, maintenance schedule, etc.).
# For now, we'll initialize it to 1 (all ATMs need service) for demonstration purposes.
status["Needs_Service"] = 1

targets = status[status["Needs_Service"] == 1].copy()

# Replace comma with period for decimal separator and then ensure numeric coords
targets["Lat"] = targets["Lat"].str.replace(',', '.', regex=True)
targets["Lon"] = targets["Lon"].str.replace(',', '.', regex=True)
targets["Lat"] = pd.to_numeric(targets["Lat"], errors="coerce")
targets["Lon"] = pd.to_numeric(targets["Lon"], errors="coerce")

targets = targets.dropna(subset=["Lat", "Lon"]).copy()

In [ ]:
# convert to radians for haversine
coords_rad = np.radians(targets[["Lat", "Lon"]].values)

eps_km = 0.7               # 🔥 tune this (0.3–1.5 typical)
earth_radius_km = 6371.0
eps = eps_km / earth_radius_km

db = DBSCAN(eps=eps, min_samples=1, metric="haversine")
labels = db.fit_predict(coords_rad)

targets["Cluster_ID"] = labels

In [ ]:
routes = []

for cid, group in targets.groupby("Cluster_ID"):
    route = {
        "Cluster_ID": int(cid),
        "Nodes": list(group["ATM_ID"]),
        "Types": list(group["ATM_Type"]),
        "Center_Lat": group["Lat"].mean(),
        "Center_Lon": group["Lon"].mean(),
        "Count": len(group)
    }
    routes.append(route)

print("\n=== CLUSTERED ROUTES (DISTANCE-BASED) ===")
for r in routes:
    print(r)


=== CLUSTERED ROUTES (DISTANCE-BASED) ===
{'Cluster_ID': 0, 'Nodes': ['ATM0001'], 'Types': ['Cash Dispenser'], 'Center_Lat': np.float64(38.497961), 'Center_Lon': np.float64(63.759627), 'Count': 1}
{'Cluster_ID': 1, 'Nodes': ['ATM0002'], 'Types': ['Cash Dispenser'], 'Center_Lat': np.float64(38.973408), 'Center_Lon': np.float64(63.891714), 'Count': 1}
{'Cluster_ID': 2, 'Nodes': ['ATM0003'], 'Types': ['Cash Dispenser'], 'Center_Lat': np.float64(38.467752), 'Center_Lon': np.float64(64.309375), 'Count': 1}
{'Cluster_ID': 3, 'Nodes': ['ATM0004'], 'Types': ['Full Function'], 'Center_Lat': np.float64(39.847521), 'Center_Lon': np.float64(61.945918), 'Count': 1}
{'Cluster_ID': 4, 'Nodes': ['ATM0005'], 'Types': ['Cash Dispenser'], 'Center_Lat': np.float64(38.640278), 'Center_Lon': np.float64(62.571616), 'Count': 1}
{'Cluster_ID': 5, 'Nodes': ['ATM0006'], 'Types': ['Full Function'], 'Center_Lat': np.float64(40.08952), 'Center_Lon': np.float64(67.520819), 'Count': 1}
{'Cluster_ID': 6, 'Nodes': ['A

In [ ]:
forecast_days = 3

predictions = []

for node_id, df in atm_history.groupby("ID"):
    df = df.sort_values("Oper_Day").copy()

    last_balance = df["Cash_Level_EOD_UZS"].iloc[-1]

    # recent behavior (last 5 days)
    recent = df.tail(5)

    avg_withdraw = recent["Withdrawals_UZS"].mean()
    avg_deposit = recent["Deposits_UZS"].mean()

    # simple trend (difference)
    trend = recent["Withdrawals_UZS"].diff().mean()
    if pd.isna(trend):
        trend = 0

    future_balance = last_balance

    for day in range(1, forecast_days + 1):
        predicted_withdraw = avg_withdraw + trend * day
        predicted_deposit = avg_deposit

        net_flow = predicted_deposit - predicted_withdraw

        future_balance += net_flow

        predictions.append({
            "ID": node_id,
            "Day_Ahead": day,
            "Pred_Withdrawals": int(predicted_withdraw),
            "Pred_Deposits": int(predicted_deposit),
            "Pred_Balance": int(future_balance)
        })

pred_df = pd.DataFrame(predictions)
print(pred_df.head())

        ID  Day_Ahead  Pred_Withdrawals  Pred_Deposits  Pred_Balance
0  ATM0001          1          12977934         870000      65155221
1  ATM0001          2          12788562         870000      53236659
2  ATM0001          3          12599190         870000      41507470
3  ATM0002          1          19836845         726959      50305639
4  ATM0002          2          20495677         726959      30536920


In [ ]:
# merge with registry for limits
pred_df = pred_df.merge(atm_registry, left_on="ID", right_on="ATM_ID", how="left")

# add kassa limits
pred_df = pred_df.merge(
    kassa_registry[["KASSA_ID", "Min_Cash_Limit", "Max_Cash_Limit"]],
    left_on="ID", right_on="KASSA_ID", how="left"
)

def detect_risk(row):
    if row["ATM_Type"] == "Cash Dispenser" or row["ATM_Type"] == "Full Function": # ATM types
        if row["Pred_Balance"] <= 0:
            return "CRITICAL_EMPTY"
        elif row["Pred_Balance"] < row["Max_Capacity_UZS"] * 0.2:
            return "LOW_CASH"
        else:
            return "OK"
    else:  # KASSA (if ID is not an ATM_ID, it means it must be KASSA_ID from the merge)
        if row["Pred_Balance"] <= row["Min_Cash_Limit"]:
            return "SHORTAGE"
        elif row["Pred_Balance"] >= row["Max_Cash_Limit"]:
            return "EXCESS"
        else:
            return "OK"

pred_df["Risk"] = pred_df.apply(detect_risk, axis=1)

print(pred_df.head())

        ID  Day_Ahead  Pred_Withdrawals  Pred_Deposits  Pred_Balance   ATM_ID  \
0  ATM0001          1          12977934         870000      65155221  ATM0001   
1  ATM0001          2          12788562         870000      53236659  ATM0001   
2  ATM0001          3          12599190         870000      41507470  ATM0001   
3  ATM0002          1          19836845         726959      50305639  ATM0002   
4  ATM0002          2          20495677         726959      30536920  ATM0002   

     Region        Lat        Lon  Max_Capacity_UZS        ATM_Type KASSA_ID  \
0  Toshkent  38,497961  63,759627         100000000  Cash Dispenser      NaN   
1  Toshkent  38,497961  63,759627         100000000  Cash Dispenser      NaN   
2  Toshkent  38,497961  63,759627         100000000  Cash Dispenser      NaN   
3   Fargona  38,973408  63,891714         150000000  Cash Dispenser      NaN   
4   Fargona  38,973408  63,891714         150000000  Cash Dispenser      NaN   

   Min_Cash_Limit  Max_Cash_Limi

In [ ]:
# only risky nodes
alerts = pred_df[pred_df["Risk"] != "OK"]

# take earliest risk per node
alerts = alerts.sort_values(["ID", "Day_Ahead"]).groupby("ID").first().reset_index()

# The required 'ATM_Type' column is already available in 'alerts' from previous merges.
# The 'Type' column in the print statement should be 'ATM_Type'.

print("\n=== ALERTS ===")
print(alerts[["ID", "ATM_Type", "Day_Ahead", "Risk"]])


=== ALERTS ===
         ID        ATM_Type  Day_Ahead      Risk
0   ATM0002  Cash Dispenser          3  LOW_CASH
1   ATM0003  Cash Dispenser          1  LOW_CASH
2   ATM0004   Full Function          3  LOW_CASH
3   ATM0006   Full Function          1  LOW_CASH
4   ATM0011  Cash Dispenser          3  LOW_CASH
5   ATM0013   Full Function          3  LOW_CASH
6   ATM0014  Cash Dispenser          2  LOW_CASH
7   ATM0015   Full Function          1  LOW_CASH
8   ATM0018   Full Function          3  LOW_CASH
9   ATM0019   Full Function          1  LOW_CASH
10  ATM0020   Full Function          1  LOW_CASH
11  ATM0021   Full Function          2  LOW_CASH
12  ATM0022  Cash Dispenser          1  LOW_CASH
13  ATM0023   Full Function          1  LOW_CASH
14  ATM0024   Full Function          1  LOW_CASH
15  ATM0025  Cash Dispenser          1  LOW_CASH
16  ATM0026  Cash Dispenser          1  LOW_CASH
17  ATM0029  Cash Dispenser          3  LOW_CASH
18  ATM0030   Full Function          3  LOW_CASH
19  

In [ ]:
pip install prophet

In [ ]:
import pandas as pd
from prophet import Prophet
import numpy as np

def preprocess_data(atm_registry, atm_history, kassa_registry, kassa_history):
    """Loads, standardizes, adds types, and merges datasets."""

    # 1. Standardize IDs and add Type column for registry data
    atm_registry_clean = atm_registry.copy()
    atm_registry_clean.rename(columns={'ATM_ID': 'ID'}, inplace=True)
    atm_registry_clean['Type'] = 'ATM'

    kassa_registry_clean = kassa_registry.copy()
    kassa_registry_clean.rename(columns={'KASSA_ID': 'ID'}, inplace=True)
    kassa_registry_clean['Type'] = 'KASSA'

    unified_registry = pd.concat([atm_registry_clean, kassa_registry_clean], ignore_index=True)

    # 2. Standardize IDs and add Type column for history data
    atm_history_clean = atm_history.copy()
    atm_history_clean['Type'] = 'ATM'

    kassa_history_clean = kassa_history.copy()
    kassa_history_clean.rename(columns={'KASSA_ID': 'ID'}, inplace=True)
    kassa_history_clean['Type'] = 'KASSA'

    unified_history = pd.concat([atm_history_clean, kassa_history_clean], ignore_index=True)

    # Convert Oper_Day to datetime
    unified_history['Oper_Day'] = pd.to_datetime(unified_history['Oper_Day'])

    return unified_registry, unified_history

def prophet_forecast(history_df, regressors, forecast_days=3, min_history_rows=10):
    """Forecasts Withdrawals_UZS using Prophet for each ID."""

    all_forecasts = []

    for unit_id in history_df['ID'].unique():
        df_unit = history_df[history_df['ID'] == unit_id].copy()

        # Skip nodes with insufficient history
        if len(df_unit) < min_history_rows:
            continue

        # Prepare data for Prophet
        df_unit = df_unit.rename(columns={'Oper_Day': 'ds', 'Withdrawals_UZS': 'y'})
        df_unit['ds'] = pd.to_datetime(df_unit['ds'])

        # Initialize and fit Prophet model
        model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)

        # Add regressors
        for regressor in regressors:
            if regressor in df_unit.columns:
                model.add_regressor(regressor)
            else:
                print(f"Warning: Regressor '{regressor}' not found for unit '{unit_id}'")

        # Handle missing regressor values in history (fill with 0 or a suitable default)
        for regressor in regressors:
            if regressor in df_unit.columns and df_unit[regressor].isnull().any():
                df_unit[regressor].fillna(0, inplace=True) # Assuming 0 is a safe default for binary flags

        model.fit(df_unit)

        # Create future dataframe
        future = model.make_future_dataframe(periods=forecast_days)

        # Add future regressors (assuming 0 for holidays/salary/ramadan for future days if no calendar provided)
        future['Is_Weekend'] = future['ds'].apply(lambda x: 1 if x.dayofweek >= 5 else 0)
        for regressor in regressors:
            if regressor not in ['Is_Weekend']:
                 # If regressor is in original df, copy its latest value, otherwise default to 0
                if regressor in df_unit.columns:
                    future[regressor] = 0 # Simplified: assume no holidays/salary days in forecast unless specified
                else:
                    future[regressor] = 0

        # Make prediction
        forecast = model.predict(future)

        # Filter for future dates only and select relevant columns
        forecast_future = forecast[forecast['ds'] > df_unit['ds'].max()].copy()
        forecast_future['ID'] = unit_id
        forecast_future.rename(columns={'ds': 'Oper_Day', 'yhat': 'Pred_Withdrawals'}, inplace=True)
        forecast_future['Pred_Withdrawals'] = forecast_future['Pred_Withdrawals'].clip(lower=0) # Withdrawals cannot be negative
        all_forecasts.append(forecast_future[['ID', 'Oper_Day', 'Pred_Withdrawals'] + regressors].copy())

    if not all_forecasts:
        return pd.DataFrame()
    return pd.concat(all_forecasts, ignore_index=True)

def simulate_balance(forecast_df, history_df, forecast_days=3, avg_deposit_period=5):
    """Estimates future balance based on forecasted withdrawals and average deposits."""

    final_predictions = []

    for unit_id in forecast_df['ID'].unique():
        df_unit_hist = history_df[history_df['ID'] == unit_id].sort_values('Oper_Day')
        df_unit_forecast = forecast_df[forecast_df['ID'] == unit_id].sort_values('Oper_Day')

        if df_unit_hist.empty or df_unit_forecast.empty:
            continue

        last_balance = df_unit_hist['Cash_Level_EOD_UZS'].iloc[-1]
        avg_deposit = df_unit_hist['Deposits_UZS'].tail(avg_deposit_period).mean()

        current_balance = last_balance

        for i, row in df_unit_forecast.iterrows():
            day_ahead = (row['Oper_Day'] - df_unit_hist['Oper_Day'].max()).days
            if day_ahead <= 0: # Ensure we only process future days
                continue

            pred_withdrawals = row['Pred_Withdrawals']
            pred_deposits = avg_deposit

            current_balance = current_balance + pred_deposits - pred_withdrawals

            final_predictions.append({
                'ID': unit_id,
                'Day_Ahead': day_ahead,
                'Pred_Withdrawals': int(pred_withdrawals),
                'Pred_Deposits': int(pred_deposits),
                'Pred_Balance': int(current_balance)
            })

    return pd.DataFrame(final_predictions)

def detect_risk(balance_predictions_df, unified_registry_df):
    """Detects risk categories for ATMs and KASSAs."""

    # Merge with registry to get max capacity and limits
    df_merged = balance_predictions_df.merge(
        unified_registry_df[['ID', 'Type', 'Max_Capacity_UZS', 'Min_Cash_Limit', 'Max_Cash_Limit']],
        on='ID', how='left'
    )

    def apply_risk_rules(row):
        if row['Type'] == 'ATM':
            if pd.isna(row['Max_Capacity_UZS']):
                return 'UNKNOWN_CAPACITY' # Handle cases where ATM capacity is missing
            if row['Pred_Balance'] <= 0:
                return 'CRITICAL_EMPTY'
            elif row['Pred_Balance'] < row['Max_Capacity_UZS'] * 0.2:
                return 'LOW_CASH'
            else:
                return 'OK'
        elif row['Type'] == 'KASSA':
            if pd.isna(row['Min_Cash_Limit']) or pd.isna(row['Max_Cash_Limit']):
                return 'UNKNOWN_LIMITS' # Handle cases where KASSA limits are missing
            if row['Pred_Balance'] <= row['Min_Cash_Limit']:
                return 'SHORTAGE'
            elif row['Pred_Balance'] >= row['Max_Cash_Limit']:
                return 'EXCESS'
            else:
                return 'OK'
        return 'UNKNOWN_TYPE'

    df_merged['Risk'] = df_merged.apply(apply_risk_rules, axis=1)
    return df_merged

# --- Main execution flow ---

# 1. Preprocess data
unified_registry, unified_history = preprocess_data(atm_registry, atm_history, kassa_registry, kassa_history)

# 2. Define regressors and forecast
prophet_regressors = ['Is_Weekend', 'Is_Salary_Day', 'Is_Holiday', 'Is_Ramadan']
forecast_output = prophet_forecast(unified_history, prophet_regressors)

# 3. Simulate balance (merge forecast with historical data for last balance/avg deposit)
# To make simulate_balance work correctly, we need the original full history_df
# with the type information.

# Merge forecast results with unified_history to pass to simulate_balance correctly
# Create a temporary df for simulation by joining forecast with the history data needed
# This is a bit complex as simulate_balance needs the last balance and average deposit from history
# and the predicted withdrawals from forecast_output.

# First, prepare a dataframe that combines historical and forecasted days for each ID,
# so simulate_balance can get last balance and current forecast.

# For simplicity, let's refine simulate_balance to take the history df and the prophet output
# and merge within if needed.

balance_predictions = simulate_balance(forecast_output, unified_history, forecast_days=3, avg_deposit_period=5)

# 4. Detect risk
final_output_df = detect_risk(balance_predictions, unified_registry)

# 5. Generate alerts
alerts_df = final_output_df[final_output_df['Risk'] != 'OK'].copy()
alerts_df = alerts_df.sort_values(['ID', 'Day_Ahead']).groupby('ID').first().reset_index()

# Removed redundant merge: alerts_df already has 'Type' from final_output_df

# --- Print Deliverables ---
print("\n--- Final Output Dataframe (first 10 rows) ---")
print(final_output_df[['ID', 'Type', 'Day_Ahead', 'Pred_Withdrawals', 'Pred_Deposits', 'Pred_Balance', 'Risk']].head(10))

print("\n--- Alerts Dataframe (earliest alert per ID) ---")
print(alerts_df[['ID', 'Type', 'Day_Ahead', 'Risk']].head(10))

/tmp/ipykernel_3454/3558166977.py:63: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_unit[regressor].fillna(0, inplace=True) # Assuming 0 is a safe default for binary flags
INFO:prophet:n_changepoints greater than number of observations. Using 18.



--- Final Output Dataframe (first 10 rows) ---
        ID Type  Day_Ahead  Pred_Withdrawals  Pred_Deposits  Pred_Balance  \
0  ATM0001  ATM          1          10466183         870000      67666972   
1  ATM0001  ATM          2          10459236         870000      58077737   
2  ATM0001  ATM          3          10688270         870000      48259467   
3  ATM0002  ATM          1          16986726         726959      53155759   
4  ATM0002  ATM          2          17026205         726959      36856512   
5  ATM0002  ATM          3          17866664         726959      19716806   
6  ATM0003  ATM          1          17176981        1390962      32790714   
7  ATM0003  ATM          2          16002538        1390962      18179138   
8  ATM0003  ATM          3          17691363        1390962       1878737   
9  ATM0004  ATM          1           6244820         748096      28743626   

       Risk  
0        OK  
1        OK  
2        OK  
3        OK  
4        OK  
5  LOW_CASH  
6     